# Text-to-Math Agent

This notebook builds a small Text-to-Math agent step by step using LangChain, Ollama, and a calculator tool.

The goal is to understand each part of the agent instead of starting with a large application.

## Task 1 - Understanding Text-to-Math Agents

**1. What is a Text-to-Math problem?**

A Text-to-Math problem is a natural language question that describes a mathematical problem. The agent has to understand the words and turn them into mathematical operations.

**2. Why are agents useful for math reasoning?**

Agents can understand a problem, decide when a calculator is useful, use the calculator, and then explain the result.

**3. Difference between a normal LLM response and an agent-based response**

- A normal LLM mainly produces an answer from its language-model knowledge.
- An agent can choose and use an external tool.
- For arithmetic, a calculator tool can perform the actual calculation.
- This is useful for multi-step problems where calculations should be checked.

## 1. Imports

First I import only the packages needed for the model, agent, and calculator.

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools import Tool
import numexpr
import re


## 2. Connect to Ollama

The model is running locally through Ollama. I use temperature 0 so that the model gives more consistent results for the same problem.

In [ ]:
llm = ChatOllama(model="llama3.2", temperature=0)

### Test the model before creating an agent

This checks that Ollama itself is working.

In [ ]:
response = llm.invoke("What is 5 + 7?")
print(response.content)

## 3. Create a calculator

The language model understands natural language, but `numexpr` performs the arithmetic. The calculator receives a mathematical expression such as `25 * 4`.

In [ ]:
def calculator(expression):
    expression = expression.strip()

    # Convert simple percentage expressions into normal arithmetic.
    expression = re.sub(r'(\d+(?:\.\d+)?)\s*%\s*of\s*(\d+(?:\.\d+)?)',
                        r'(\1 / 100) * \2', expression, flags=re.IGNORECASE)

    expression = expression.replace('%', '/100')
    return str(numexpr.evaluate(expression))

print("5 + 10 =", calculator("5 + 10"))
print("12.5 * 4 =", calculator("12.5 * 4"))
print("15% of 240 =", calculator("15% of 240"))

In [ ]:
print("5 + 10 =", calculator("5 + 10"))
print("12.5 * 4 =", calculator("12.5 * 4"))
print("15 / 100 * 240 =", calculator("15 / 100 * 240"))

## 4. Turn the calculator into a LangChain tool

The agent cannot directly call my Python function unless it is exposed as a tool. The tool name and description tell the agent what the calculator is for.

The calculator expects an expression, not a sentence. For example, a percentage must first be represented mathematically as `15 / 100 * 240`.

In [ ]:
math_tool = Tool(
    name="Calculator",
    func=calculator,
    description=(
        "Use this tool for arithmetic. Give it a mathematical expression. "
        "It supports decimals and simple percentage expressions such as "
        "15% of 240."
    )
)

## 5. Create the agent

The agent combines the Ollama model and the calculator. The model decides when the calculator is needed and then uses the result to produce the answer.

In [ ]:
agent = initialize_agent(
    tools=[math_tool],
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=8,
    handle_parsing_errors=True,
    early_stopping_method="generate",
)


## 6. Basic arithmetic test

The first agent test is intentionally simple.

In [ ]:
question = "Calculate 25 + 15. Use the Calculator tool and then give the final answer."
answer = agent.run(question)
print(answer)

## 7. Algebra test

The agent should be able to understand a simple algebra problem and use arithmetic when required.

In [ ]:
question = "Solve x + 5 = 12. Explain the algebra steps."
answer = agent.run(question)
print(answer)

print("\nNote: this is a reasoning test. The Calculator tool is for arithmetic; it is not a symbolic algebra solver.")

## 8. Decimal tests

Decimals were one of the problem areas in the original submission, so I test them separately.

In [ ]:
decimal_questions = [
    "What is 12.5 multiplied by 4?",
    "What is 15.75 plus 3.25?",
    "What is 7.5 divided by 2.5?"
]

for question in decimal_questions:
    print("Problem:", question)
    print("Answer:", agent.run(question))
    print()

## 9. Percentage tests

Percentages need an extra interpretation step. For example, 15% means 15 divided by 100. I test both a direct percentage and a discount problem.

In [ ]:
percentage_questions = [
    "Calculate 15% of 240 using the Calculator tool.",
    "A product costs 800 and has a 20% discount. Calculate the discount and then the final price.",
    "Calculate a 10% increase on 500."
]

for question in percentage_questions:
    print("Problem:", question)
    print("Answer:", agent.run(question))
    print()

## 10. Multi-step problem

This test checks whether the agent can break a word problem into more than one calculation.

In [ ]:
question = "A shirt costs 800. It has a 15% discount. After the discount, add 50 shipping. What is the final amount? Solve step by step."
answer = agent.run(question)
print(answer)

## 11. Preserve mathematical context

The original Streamlit program displayed history, but displaying history does not automatically give that history to the agent. Here I explicitly include the previous result in the next prompt.

In [ ]:
first_question = "Calculate 20% of 500 using the Calculator tool."
first_answer = agent.run(first_question)
print("First answer:", first_answer)

follow_up = f"""
Previous problem: {first_question}
Previous answer: {first_answer}

Follow-up: add 25 to the numerical result from the previous answer.
Use the Calculator tool.
"""

second_answer = agent.run(follow_up)
print("Follow-up answer:", second_answer)

## 12. Final simple function

This function is the part that can later be connected to a Streamlit interface. It keeps the agent setup separate from the user interface.

In [ ]:
def solve_math(question, previous_context=""):
    instruction = (
        "Use the Calculator tool for arithmetic. "
        "Give the final answer as a plain number with no currency symbol or units "
        "unless the question specifically asks about money."
    )
    if previous_context:
        prompt = (
            f"Previous context:\n{previous_context}\n\n"
            f"New problem:\n{question}\n"
            f"{instruction}"
        )
    else:
        prompt = f"{question}\n{instruction}"

    return agent.run(prompt)

def clean_answer(answer):
    return answer.replace("$", "").strip()

## 13. Final test

I now test the final function with a percentage problem.

In [ ]:
print(clean_answer(solve_math("Calculate 25% of 320 using the Calculator tool.")))

## Troubleshooting

If `langchain.agents` or `initialize_agent` is not available in your installed LangChain version, the package versions are different from the versions used for the original script. In that case, install the versions from the assignment environment or update the agent code to the current LangChain API.

If Ollama reports that `llama3.2` is missing, run `ollama list` in Terminal and use a model that is installed locally.

If the model is not running, start Ollama before executing the model cells.

## Conclusion

The agent has three main parts: the Ollama language model, the calculator tool, and the LangChain agent that decides when to use the tool.

Testing percentages and decimals separately is important because the calculator can perform those calculations once they are represented as valid mathematical expressions. Context also has to be passed explicitly; simply storing previous answers in a user-interface history does not give the agent access to them.